In [3]:
from langchain_core.messages import HumanMessage
from langgraph.graph.message import MessagesState
from langgraph.graph import START,END,StateGraph
from langchain_deepseek import ChatDeepSeek

from dotenv import load_dotenv
load_dotenv(override=True)
#0. 链接LLM
model = ChatDeepSeek(
    model="deepseek-v4-flash",
    extra_body={
        "thinking":
            {"type":"disabled"}

    }
)

class OverAllState(MessagesState):
    username:str
    output:str
#1. 创建图
def node_a(state:OverAllState)->OverAllState:
  return{
      "messages":[HumanMessage("你好，我是"+state["username"])],
  }

def llm_node(state:OverAllState) ->OverAllState:
    res = model.invoke(state["messages"])
    return {
        "messages":[res],
        "output":res.content
    }

builder = StateGraph(state_schema=OverAllState)
builder.add_node("node_a",node_a)
builder.add_node("llm_node",llm_node)
builder.add_edge(START,"node_a")
builder.add_edge("node_a","llm_node")
builder.add_edge("llm_node",END)

graph = builder.compile()
res = graph.invoke({"username":"张三"})
print(res)




{'messages': [HumanMessage(content='你好，我是张三', additional_kwargs={}, response_metadata={}, id='bef785be-350e-44f4-b1c0-5a9bc94371a8'), AIMessage(content='你好，张三！很高兴认识你。😊\n\n有什么我可以帮你的吗？无论是学习、工作、生活上的问题，还是想聊聊天，我都在这儿呢。', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 8, 'total_tokens': 43, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 8}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_a18b46594c_prod0820_fp8_kvcache_20260402', 'id': 'dd8cd07b-2a20-4c9c-b718-5a9f8e1cf889', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019fe9f1-e75d-74a0-b267-f6c6e0477b65-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 35, 'total_tokens': 43, 'input_token_details': {'cache_read': 0}, 'output_token_details